# 人間による介入（Human-in-the-Loop）：事前アクションゲート、リスク分類、監査ログ

このレッスンのREADMEでは、人間による介入を紹介し、エージェントが応答を生成した後にユーザーに `APPROVE` または `REJECT` を求める簡単なスニペットを紹介しています。このパターンは良い出発点ですが、実運用のHITL実装では通常、以下の3つの追加要素が必要です。

1. エージェントがリスクのあるステップを実行する<strong>前に</strong>動作する<strong>事前アクションゲート</strong>。これによりコストや不可逆性、遅延が制御されます。
2. <strong>リスク分類</strong>。低リスクのアクションは自動実行、中リスクはバッチ承認、高リスクのみ人間の判断が介在します。
3. <strong>監査ログと修正ループ</strong>。すべてのゲート判定をJSONL形式で記録し、拒否時には単に「Revising...」と表示するのではなく、構造化された理由とともにエージェントに再提示します。

このノートブックではこれらを `06-system-message-framework.ipynb` と同じプリミティブの上に構築しています。`DEMO_MODE = True` では対話入力不要のエンドツーエンド実行が可能で、`DEMO_MODE = False` では実際の `input()` プロンプトが使用されます。なお、DEMO_MODEでは3つめの目標のリトライがスクリプト化されており、ループの動作がエンドツーエンドで見えるようになっています。実際の修正駆動再分類には `DEMO_MODE = False` とオペレーターが必要です。

**対象外（他のレッスンで扱う）：** 認証とアクセス制御（Lesson 06 README 脅威2）、ツール呼び出しミドルウェア（Lesson 14 MAF詳細）、マルチエージェント討論パターン。

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## パターン1: 事前アクションゲート

READMEのHITLスニペットは、まずエージェントを呼び出し、その後ユーザーに出力の承認を求めます。これは<strong>事後アクション</strong>のフローです。エージェントはすでに実行されているため、LLM呼び出しのコストは支払われており、任意の副作用（送信されたメール、書き込まれたデータベースの行、投稿されたコメント）もすでに発生しています。

<strong>事前アクション</strong>のフローは、リスクのあるステップの前にゲートを挿入します。エージェントがアクションを提案し、ゲートが実行するかどうかを判断し、承認があって初めて副作用が発生します。

| 観点 | 事後承認 (READMEスニペット) | 事前アクションゲート (本ノートブック) |
|---|---|---|
| 承認はいつ行われるか？ | エージェントが出力を生成した後 | いかなる副作用が実行される前 |
| 拒否時のLLMコスト | すでに支払済み | 提案部分のみ支払い、実行部分は支払わない |
| 取り消し不可能な副作用 | 発生しうる（アクションはすでに起きている） | 防止される |
| 監査の明確さ | 承認はプリント文である | 承認はタイムスタンプ、アクション、理由を含むJSON記録である |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## パターン 2: リスクの階層化

すべての操作に人間の承認が必要なわけではありません。公開APIに対する読み取り専用の参照は、顧客へのメール送信とはリスクが異なります。両方を同じ扱いにすると、オペレーターの注意が浪費され、エージェントの動作が遅くなります。

シンプルな3段階モデル：

| Tier | 例 | 承認フロー |
|---|---|---|
| `low` (読み取り専用) | ナレッジベースの検索、フライトオプションの参照、公開ウェブページの取得 | 自動実行、監査用にログ記録 |
| `medium` (低コストな変更) | 結果をキャッシュ、カウンターを増加、リマインダーをスケジュール | 自動実行、ただし日次の一括レビューあり |
| `high` (外部向けまたは不可逆的) | メール送信、カード請求、公開チャンネルへの投稿 | 人間の承認でブロック |

これは一例の階層です。運用システムでは、より細かい階層（例：AWS IAMの権限レベル、役割ベースのアクセス階層）を使うことが多いです。以下の3段階版は、読み取り専用と副作用を伴う操作を混在させるエージェントにとって最も小さい実用的なバージョンです。

以下の分類器はキーワードのヒューリスティックを使っており、デモが決定論的かつ安価に動くようにしています。運用システムでは、学習済み分類器やポリシーエンジンに置き換えます。

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## パターン 3: 監査ログとリビジョンループ

`print("Response approved.")` は監査ログではありません。信頼性のために、すべてのゲートの判断は構造化されたイベントとして記録されるべきであり、後でクエリを実行したり、再生したり、インシデントレビューに添付したりできるようにする必要があります。

2つの要素:

1. **追記専用の JSONL。** 判断ごとに1行で、タイムスタンプ、アクション、階層、判断、理由を含む。grepが簡単で、後で実際のログストアに送るのも容易。
2. **否認時のリビジョンループ。** ゲートが `deny` を返したとき、エージェントは拒否理由を文脈として自身に再プロンプトをかけるので、次の提案で問題を回避できる。

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 追加リソース

これらのHITLパターンのバリエーションを実装した他の公開プロジェクトもいくつかあります。アプローチを比較して、あなたのスタックに合ったものを見つけてください。

- **LangChain** のヒューマンインザループツールラッパー ([docs](https://python.langchain.com/docs/integrations/tools/human_tools))：ヒューマン入力のために実行を一時停止するドロップインのツールラッパー。
- **AutoGen** の `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ではこれが再構成されています)：マルチエージェント会話でヒューマンを表現するためのエージェントロールを使用。
- **Semantic Kernel** の関数フィルター ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters))：すべての関数呼び出しの前後で実行されるミドルウェアで、ゲーティングロジックに適しています。

各プロジェクトは3つのサブパターンを異なる方法で扱っています。LangChainはツールとしてラップし、AutoGenはエージェントロールを使い、Semantic Kernelはミドルウェアフィルターを使います。自分のエージェントの設計を選ぶ前に、1つか2つの実装を最初から最後まで読んでみてください。

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
